<a href="https://colab.research.google.com/github/matiasmorgan1/Master-s-by-Research/blob/main/Figure_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Create individual plots for each glacier, colour-coded and grouped by sensitivity regime. Plots include CMEMS ocean temperature data at different depths, MAR atmopsheric temperature data split into seasons, and ice discharge data from Mankoff et al. (2020). All data has been collected according to the methodology in the thesis associated with this code.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

# 2. Re-mount cleanly
from google.colab import drive
drive.mount('/content/drive')

# ==============================================
# Paths
# ==============================================
file_path = "/content/drive/My Drive/Spreadsheets (3)/MASTER_SPREADSHEET_SEASONAL.xlsx"
output_dir = "/content/drive/My Drive/Spreadsheets (3)/GlacierFigures"
os.makedirs(output_dir, exist_ok=True)

# ==============================================
# Cluster Mapping & Colors
# ==============================================
cluster_dict = {
    # Cluster 1 (10 glaciers)
    232: 1, 245: 1, 296: 1, 266: 1, 284: 1, 293: 1, 304: 1, 300: 1, 335: 1, 310: 1,
    # Cluster 2 (6 glaciers)
    303: 2, 320: 2, 302: 2, 330: 2, 281: 2, 258: 2,
    # Cluster 3 (2 glaciers)
    298: 3, 297: 3,
    # Cluster 4 (2 glaciers)
    233: 4, 224: 4,
    # Cluster 5 (2 glaciers)
    227: 5, 226: 5,
    # Cluster 6 (3 glaciers)
    238: 6, 239: 6, 270: 6
}

group_color_map = {
    1: "#4776EE",
    2: "#1BD0D5",
    3: "#61FC6C",
    4: "#D2E935",
    5: "#FE9B2D",
    6: "#DA3907"
}

# Omitted glacier summing map {target_glacier_id: omitted_glacier_id}
omitted_map = {
    284: 286,
    320: 323
}

# ==============================================
# Updated Layout Map (row_idx, col_idx)
# ==============================================
glacier_positions = {
    # Row 0: Group 1 (First 5 glaciers)
    232: (0, 0), 245: (0, 1), 296: (0, 2), 266: (0, 3), 284: (0, 4),

    # Row 1: Group 1 (Final 5 glaciers)
    293: (1, 0), 304: (1, 1), 300: (1, 2), 335: (1, 3), 310: (1, 4),

    # Row 2: Group 2 (First 4 glaciers)
    303: (2, 0), 320: (2, 1), 302: (2, 2), 330: (2, 3),

    # Row 3: Group 2 (Last 2 glaciers), Col Gap (col 2), Group 3 (Both glaciers)
    281: (3, 0), 258: (3, 1), 297: (3, 3), 298: (3, 4),

    # Row 4: Group 4 (Both glaciers), Col Space (col 2), Group 5 (Both glaciers)
    224: (4, 0), 233: (4, 1), 226: (4, 3), 227: (4, 4),

    # Row 5: Group 6 (All 3 glaciers)
    238: (5, 0), 239: (5, 1), 270: (5, 2)
}

# ==============================================
# Load Excel Sheets & Build ID Mapping
# ==============================================
xls = pd.ExcelFile(file_path)
glacier_names = xls.sheet_names

# Create lookups between Glacier IDs and sheet names
id_to_sheet = {}
for name in glacier_names:
    try:
        g_id = int(''.join(filter(str.isdigit, name)))
        id_to_sheet[g_id] = name
    except ValueError:
        continue

# Map structural positioning data to target glacier lists per row
row_to_glaciers = {r: [] for r in range(6)}
for glacier_id, (r_idx, c_idx) in glacier_positions.items():
    if glacier_id in id_to_sheet:
        glacier_name = id_to_sheet[glacier_id]
        row_to_glaciers[r_idx].append((c_idx, glacier_name, glacier_id))

# Sort each row list by column index to maintain ordered placement
for r in row_to_glaciers:
    row_to_glaciers[r].sort(key=lambda x: x[0])

# Max grid columns needed across rows
n_macro_cols = 5

# ==============================================
# Loop Row by Row (Each Row saves a standalone Figure)
# ==============================================
for row_idx in range(6):
    glaciers_in_row = row_to_glaciers[row_idx]
    if not glaciers_in_row:
        continue  # Skip row processing if no targets mapped

    fig = plt.figure(figsize=(30, 6.66), constrained_layout=True)
    subfigures = fig.subfigures(1, n_macro_cols)

    for col_idx, glacier_name, glacier_id in glaciers_in_row:

        subfig = subfigures[col_idx] if n_macro_cols > 1 else subfigures

        # --- Extract grouping metrics ---
        group_id = cluster_dict[glacier_id]
        current_color = group_color_map[group_id]

        # --- Load primary dataset sheet ---
        df = pd.read_excel(file_path, sheet_name=glacier_name)
        df["Date"] = pd.to_datetime(df["Date"])
        df = df.sort_values("Date").reset_index(drop=True)

        # --- Check and sum omitted glacier discharge if applicable ---
        if glacier_id in omitted_map:
            omitted_id = omitted_map[glacier_id]
            if omitted_id in id_to_sheet:
                omitted_sheet_name = id_to_sheet[omitted_id]
                df_omitted = pd.read_excel(file_path, sheet_name=omitted_sheet_name)
                df_omitted["Date"] = pd.to_datetime(df_omitted["Date"])

                # Merge on Date to align time steps accurately before summing
                df = pd.merge(
                    df,
                    df_omitted[["Date", "Discharge Anomaly (Gt/yr)"]],
                    on="Date",
                    how="left",
                    suffixes=("", "_omitted")
                )
                df["Discharge Anomaly (Gt/yr)"] = (
                    df["Discharge Anomaly (Gt/yr)"].fillna(0) +
                    df["Discharge Anomaly (Gt/yr)_omitted"].fillna(0)
                )

        # --- Seasonal date mapping ---
        def build_seasonal_date(row):
            year = row["Date"].year
            season = row["Season"]
            if season == "DJF": return pd.Timestamp(year=year, month=2, day=1)
            elif season == "MAM": return pd.Timestamp(year=year, month=5, day=1)
            elif season == "JJA": return pd.Timestamp(year=year, month=8, day=1)
            elif season == "SON": return pd.Timestamp(year=year, month=11, day=1)
            return row["Date"]

        df["SeasonDate"] = df.apply(build_seasonal_date, axis=1)
        season_dates = df["SeasonDate"]

        # --- Ocean columns configuration ---
        ocean_cols = [c for c in df.columns if c.startswith("Ocean Temp")]
        depths = np.array([
            (float(c.split("Ocean Temp ")[1].split("-")[0]) +
             float(c.split("Ocean Temp ")[1].split("-")[1].replace("m (°C)", ""))) / 2
            for c in ocean_cols
        ])
        ocean_matrix = df[ocean_cols].to_numpy().T

        # --- Generate 6 local subplots inside this subfigure card ---
        axes = subfig.subplots(
            6, 1, sharex=True,
            gridspec_kw={"height_ratios": [1, 1, 1, 1, 1, 2]}
        )

        # 1️⃣ Discharge Subpanel
        axes[0].plot(season_dates, df["Discharge Anomaly (Gt/yr)"], linestyle="-", color=current_color)
        axes[0].set_ylabel("Discharge\n(Gt/yr)", fontsize=12)
        axes[0].tick_params(labelbottom=False, labelsize=10)
        axes[0].spines['top'].set_visible(False)
        axes[0].spines['right'].set_visible(False)

        # 2–5️⃣ Seasonal surface temps Subpanels
        seasonal_order = ["SON", "JJA", "MAM", "DJF"]
        for i, season in enumerate(seasonal_order):
            mask = df["Season"] == season
            axes[i+1].plot(season_dates[mask], df.loc[mask, "Surface temp Anomaly (°C)"], linestyle="-", color=current_color)
            axes[i+1].set_ylabel(f"{season}\nSurf. T (°C)", fontsize=10)
            axes[i+1].spines['top'].set_visible(False)
            axes[i+1].spines['right'].set_visible(False)
            axes[i+1].tick_params(labelsize=12)
            if i != 3:
                axes[i+1].tick_params(labelbottom=False)

        # 6️⃣ Ocean Heatmap Subpanel
        time_nums = mdates.date2num(season_dates)
        dt = np.diff(time_nums).mean() if len(time_nums) > 1 else 1
        time_edges = np.concatenate([[time_nums[0]-dt/2], time_nums+dt/2])
        depth_step = np.diff(depths).mean() if len(depths) > 1 else 1
        depth_edges = np.concatenate([[depths[0]-depth_step/2], depths+depth_step/2])

        pcm = axes[5].pcolormesh(
            time_edges, depth_edges, ocean_matrix,
            cmap="RdBu_r", vmin=-2, vmax=2, shading="flat"
        )
        axes[5].invert_yaxis()
        axes[5].spines['top'].set_visible(False)
        axes[5].spines['right'].set_visible(False)
        axes[5].xaxis_date()
        axes[5].xaxis.set_major_locator(mdates.YearLocator(2))
        axes[5].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        axes[5].tick_params(labelsize=12)

        # Individual colorbar mapped cleanly into the subfigure panel
        cbar = subfig.colorbar(pcm, ax=axes[5], aspect=12, pad=0.02)
        cbar.ax.tick_params(labelsize=8)

        # Individual subfigure title
        subfig.suptitle(f"{glacier_id}", fontsize=36, fontweight='bold', y=1.07)

    # Save current row structure as a standalone file
    outpath = os.path.join(output_dir, f"glacier_mosaic_row_{row_idx}.png")
    plt.savefig(outpath, dpi=250, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ Row compilation saved completely to: {outpath}")

The following blocks of code create figures investigating similarities and differences of glacier properties and controls between and within groups.

In [1]:
# Visualisation of how similar certain properties of glaciers are within and between groups.

import os
import re
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive

# ============================================================
# 1. MOUNT & DATA LOADING
# ============================================================
drive.mount("/content/drive", force_remount=True)

file_path = "/content/drive/MyDrive/Spreadsheets/Controls_Master.xlsx"

# Load data
df = pd.read_excel(file_path, sheet_name="Sheet1")

# Clean index and column strings
df.columns = df.columns.astype(str).str.strip()
first_col = df.columns[0]
df[first_col] = df[first_col].astype(str).str.strip()
df = df.set_index(first_col)

# Explicitly map combined columns to '284' and '320'
col_rename_map = {}
for col in df.columns:
    digits = [int(d) for d in re.findall(r"\d+", col)]
    if 284 in digits and 286 in digits:
        col_rename_map[col] = "284"
    elif 320 in digits and 323 in digits:
        col_rename_map[col] = "320"

df = df.rename(columns=col_rename_map)

# ============================================================
# 2. SETUP & 6-CLUSTER MAPPINGS (25 TARGET GLACIERS)
# ============================================================
group_color_map = {
    1: "#4776EE",
    2: "#1BD0D5",
    3: "#61FC6C",
    4: "#D2E935",
    5: "#FE9B2D",
    6: "#DA3907",
}
combined_color = "#7f8c8d"  # Muted gray for background combined dataset

# 25 unique target glaciers mapped to their new clusters
cluster_dict = {
    # Cluster 1 (10 glaciers)
    232: 1,
    245: 1,
    296: 1,
    266: 1,
    284: 1,  # (286 omitted in favor of 284)
    293: 1,
    304: 1,
    300: 1,
    335: 1,
    310: 1,
    # Cluster 2 (6 glaciers)
    303: 2,
    320: 2,  # (323 omitted in favor of 320)
    302: 2,
    330: 2,
    281: 2,
    258: 2,
    # Cluster 3 (2 glaciers)
    298: 3,
    297: 3,
    # Cluster 4 (2 glaciers)
    233: 4,
    224: 4,
    # Cluster 5 (2 glaciers)
    227: 5,
    226: 5,
    # Cluster 6 (3 glaciers)
    238: 6,
    239: 6,
    270: 6,
}

# Explicit list of the exact 25 glacier IDs we want to include
TARGET_25_GLACIERS = [str(gid) for gid in cluster_dict.keys()]

fig1_controls = [
    "Bathymetric High Point depth (m)",
    "Surface Elevation 2km and 5km average (m)",
    "0-5km Average Surface Gradient (°)",
    "Fjord Length (m)",
]
fig1_titles = [
    "Bathymetric High Points",
    "Surface Elevation (2 & 5km Avg)",
    "Surface Gradient (0-5km)",
    "Fjord Length",
]

fig2_controls = [
    "Bathymetric High Point depth (m)",
    "Surface Elevation 2km along flowline (m)",
    "0-5km Average Surface Gradient (°)",
    "Fjord Length (m)",
]
fig2_titles = [
    "Bathymetric High Point depth (m)",
    "Surface Elevation 2km along flowline (m)",
    "0-5km Mean Surface Gradient (°)",
    "Fjord Length (m)",
]


# ============================================================
# 3. HELPER PLOTTING FUNCTION
# ============================================================
def plot_control_grid(control_list, title_list, figure_main_title):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    clusters = sorted(set(cluster_dict.values()))

    for idx, control in enumerate(control_list):
        ax = axes[idx]
        display_title = title_list[idx]

        if control not in df.index:
            ax.text(
                0.5,
                0.5,
                f"Control Not Found:\n'{control}'",
                ha="center",
                va="center",
                color="red",
                fontsize=12,
            )
            ax.set_title(display_title, color="gray")
            ax.set_axis_off()
            continue

        labels = []
        available_target_cols = []

        # Plotting logic for individual Clusters
        for i, cluster in enumerate(clusters):
            group_glaciers = [
                str(g) for g, c in cluster_dict.items() if c == cluster
            ]
            valid_glaciers = [g for g in group_glaciers if g in df.columns]

            available_target_cols.extend(valid_glaciers)

            values_series = pd.to_numeric(
                df.loc[control, valid_glaciers], errors="coerce"
            ).dropna()

            cluster_color = group_color_map.get(cluster, "#000000")
            x_center = i + 1

            for g_name, val in values_series.items():
                jitter = np.random.uniform(-0.15, 0.15)
                x_pos = x_center + jitter

                ax.scatter(
                    x_pos,
                    val,
                    color=cluster_color,
                    s=45,
                    zorder=3,
                    alpha=0.9,
                    edgecolors="white",
                    linewidths=0.5,
                )

            # Coefficient of Variation (CV) Calculation
            if len(values_series) > 1 and values_series.mean() != 0:
                cv = values_series.std() / values_series.mean()
                cv_text = f"CV: {cv:.2f}"
            else:
                cv_text = "CV: N/A"

            ax.text(
                x_center,
                1.01,
                cv_text,
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=8.5,
                fontweight="semibold",
                color="#555555",
            )

            if "Bathymetric" in control:
                labels.append(
                    f"Group {cluster}\n{len(values_series)}/{len(group_glaciers)}"
                )
            else:
                labels.append(f"Group {cluster}")

        # COMBINED COLUMN: Strictly query ONLY the 25 target glacier columns
        combined_valid_cols = [
            col for col in TARGET_25_GLACIERS if col in df.columns
        ]
        combined_values = pd.to_numeric(
            df.loc[control, combined_valid_cols], errors="coerce"
        ).dropna()
        combined_pos = len(clusters) + 1

        for val in combined_values:
            jitter = np.random.uniform(-0.15, 0.15)
            ax.scatter(
                combined_pos + jitter,
                val,
                color=combined_color,
                s=30,
                zorder=2,
                alpha=0.3,
            )
        labels.append(f"Combined\n(n={len(combined_values)})")

        # CV for Combined Dataset
        if len(combined_values) > 1 and combined_values.mean() != 0:
            comp_cv = combined_values.std() / combined_values.mean()
            comp_cv_text = f"CV: {comp_cv:.2f}"
        else:
            comp_cv_text = "CV: N/A"

        ax.text(
            combined_pos,
            1.01,
            comp_cv_text,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            fontsize=8.5,
            fontweight="semibold",
            color="#555555",
        )

        # Subplot Formatting
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel("Value", fontweight="bold", fontsize=10)
        ax.set_title(display_title, fontweight="bold", fontsize=14, pad=24)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    plt.subplots_adjust(hspace=0.5, wspace=0.3)
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    if figure_main_title:
        fig.suptitle(figure_main_title, fontsize=20, fontweight="bold")
    plt.show()


# ============================================================
# 4. EXECUTION
# ============================================================
print("Cleaned Index names found in spreadsheet:")
for name in df.index:
    print(f" - {name}")

plot_control_grid(
    fig1_controls, fig1_titles, "Geometric Controls Analysis: Dataset A"
)
plot_control_grid(fig2_controls, fig2_titles, "")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
#Coefficients of variation for controls in the above block of code per sensitivity regime.

# ================================
# 1. Import libraries
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================================
# 2. Controls to analyse
# ================================
# We use a list comprehension to ensure we only try to access controls that
# actually exist in your df.index and are not 'nan'
controls_to_use = [
    "Bathymetric High Point depth (m)",
    "Fjord Length (m)",
    "0-5km Average Surface Gradient (°)",
    'Surface Elevation 2km along flowline (m)'
]

# Ensure we only use controls present in the index to avoid KeyError
actual_controls = [c for c in controls_to_use if c in df.index]
df_subset = df.loc[actual_controls]

# ================================
# 3. Glacier clusters
# ================================
cluster_dict = {
    239:1, 245:1, 238:1, 286:1, 323:1, 320:1, 303:1, 300:1, 330:1,
    270:2, 258:2, 281:2, 233:2, 224:2,
    232:3, 266:3, 335:3, 310:3, 296:3, 293:3, 284:3, 302:3, 304:3,
    226:4, 227:4,
    297:5, 298:5
}

clusters = [1, 2, 3]

# ================================
# 4. Create results table
# ================================
cv_table = pd.DataFrame(index=df_subset.index.unique(), columns=[f"Group {c}" for c in clusters])

# ================================
# 5. Compute CV
# ================================
for control in cv_table.index:
    for cluster in clusters:
        # Get glacier IDs as strings to match spreadsheet columns
        glaciers = [
            str(g) for g, c in cluster_dict.items()
            if c == cluster and str(g) in df_subset.columns
        ]

        if not glaciers:
            continue

        # 1. Extract and Flatten (handles duplicates/2D data)
        raw_data = df_subset.loc[control, glaciers]
        if isinstance(raw_data, pd.DataFrame):
            flat_data = raw_data.values.flatten()
        else:
            flat_data = raw_data

        # 2. Force to numeric and drop NaNs
        values = pd.to_numeric(flat_data, errors='coerce')
        values = values[~np.isnan(values)]

        # 3. Calculate CV (Std Dev / |Mean|)
        if len(values) > 1:
            mean_val = np.mean(values)
            std_val = np.std(values, ddof=1)
            # Use abs(mean) to keep CV positive even for gradients/elevations
            cv = std_val / abs(mean_val) if mean_val != 0 else np.nan
        else:
            cv = np.nan

        cv_table.loc[control, f"Group {cluster}"] = cv

# ================================
# 6. Convert table to image
# ================================
# Final cleanup of the table data
table_display = cv_table.astype(float).round(3)

fig, ax = plt.subplots(figsize=(10, 5))  # Slightly wider for text
ax.axis("off")

table = ax.table(
    cellText=table_display.values,
    rowLabels=table_display.index,
    colLabels=table_display.columns,
    cellLoc="center",
    loc="center"
)

# Styling the table
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.0)  # Increased vertical scale for better row spacing

# Bold the headers
for (row, col), cell in table.get_celld().items():
    if row == 0 or col == -1:
        cell.set_text_props(weight='bold')

plt.title("Cluster Tightness (Coefficient of Variation)", pad=30, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Relationship between dominant ocean lag and fjord length per glacier, colour-coded by sensitivity regime.

import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ============================================================
# 1. SETUP & 6-CLUSTER MAPPINGS (UPDATED COLORS & GROUPS)
# ============================================================
group_color_map = {
    1: "#4776EE",
    2: "#1BD0D5",
    3: "#61FC6C",
    4: "#D2E935",
    5: "#FE9B2D",
    6: "#DA3907",
}

# 25 unique target glaciers mapped to their updated clusters
cluster_dict = {
    # Cluster 1 (10 glaciers)
    232: 1,
    245: 1,
    296: 1,
    266: 1,
    284: 1,
    286: 1,  # Mapped to 284 in dual headers
    293: 1,
    304: 1,
    300: 1,
    335: 1,
    310: 1,
    # Cluster 2 (6 glaciers)
    303: 2,
    320: 2,
    323: 2,  # Mapped to 320 in dual headers
    302: 2,
    330: 2,
    281: 2,
    258: 2,
    # Cluster 3 (2 glaciers)
    298: 3,
    297: 3,
    # Cluster 4 (2 glaciers)
    233: 4,
    224: 4,
    # Cluster 5 (2 glaciers)
    227: 5,
    226: 5,
    # Cluster 6 (3 glaciers)
    238: 6,
    239: 6,
    270: 6,
}

# Standardise DataFrame column names for 284/286 and 320/323 if present
if "df" in locals():
    col_rename_map = {}
    for col in df.columns:
        digits = [int(d) for d in re.findall(r"\d+", str(col))]
        if 284 in digits and 286 in digits:
            col_rename_map[col] = "284"
        elif 320 in digits and 323 in digits:
            col_rename_map[col] = "320"
    df = df.rename(columns=col_rename_map)

# ============================================================
# 2. PROCESSING PLSR RESULTS FOR POSITIVE DOMINANT OCEAN LAG
# ============================================================
exclude_cols = ["Glacier", "Fjord Depth (m)", "R2"]
loading_cols = [
    c for c in plsr_lagged_results.columns if c not in exclude_cols
]

ocean_positive_lags = []

for idx, row in plsr_lagged_results.iterrows():
    glacier_name = str(row["Glacier"]).strip()

    # Standardise glacier name for dual IDs
    digits = [int(d) for d in re.findall(r"\d+", glacier_name)]
    if 284 in digits and 286 in digits:
        glacier_name = "284"
    elif 320 in digits and 323 in digits:
        glacier_name = "320"

    lag_weights_accumulator = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
    has_positive_ocean_data = False

    for col in loading_cols:
        if "Ocean Temp" in col:
            loading_val = row[col]

            if not pd.isna(loading_val) and loading_val > 0:
                lag_match = re.search(r"_lag(\d+)", col)
                if lag_match:
                    lag_num = int(lag_match.group(1))
                    if lag_num in lag_weights_accumulator:
                        lag_weights_accumulator[lag_num] += loading_val
                        has_positive_ocean_data = True

    best_lag = (
        max(lag_weights_accumulator, key=lag_weights_accumulator.get)
        if has_positive_ocean_data
        else 0
    )

    ocean_positive_lags.append(
        {"Glacier": glacier_name, "Optimal_Lag": best_lag}
    )

df_lags = pd.DataFrame(ocean_positive_lags).drop_duplicates(
    subset=["Glacier"]
)

# ============================================================
# 3. MATCHING FJORD LENGTH AND CLUSTER ASSIGNMENTS
# ============================================================
fjord_length_row = "Fjord Length (m)"

fjord_data = []
if "df" in locals() and fjord_length_row in df.index:
    for g_col in df.columns:
        glacier_id_str = str(g_col).strip()
        val = pd.to_numeric(df.loc[fjord_length_row, g_col], errors="coerce")
        if not pd.isna(val):
            fjord_data.append({"Glacier": glacier_id_str, "Fjord_Length": val})

df_fjord = pd.DataFrame(fjord_data)

# Cluster mapping with unique 25 glaciers
cluster_mapping = [
    {"Glacier": str(g).strip(), "Cluster": c}
    for g, c in cluster_dict.items()
    if g not in [286, 323]  # Omit duplicate keys
]
df_clusters = pd.DataFrame(cluster_mapping)

# Merge datasets
plot_df = pd.merge(df_lags, df_fjord, on="Glacier", how="inner")
plot_df = pd.merge(plot_df, df_clusters, on="Glacier", how="inner")

# ============================================================
# 4. STATISTICAL CALCULATIONS (CATEGORICAL R² METRICS)
# ============================================================
print("--- CATEGORICAL STATISTICAL RESULTS (Variance Explained by Lag) ---")

# Entire Sample Global Metric
if len(plot_df["Optimal_Lag"].unique()) > 1 and len(plot_df) > 2:
    global_model = ols("Fjord_Length ~ C(Optimal_Lag)", data=plot_df).fit()
    global_r2 = global_model.rsquared
    print(f"Entire Sample (n={len(plot_df)}): Categorical R² = {global_r2:.4f}")
else:
    global_r2 = None
    print("Entire Sample: Insufficient variance/data to compute Categorical R²")

# Individual Group Variance Metrics across all 6 groups
group_r2_map = {}

for cluster_id in sorted(group_color_map.keys()):
    cluster_subset = plot_df[plot_df["Cluster"] == cluster_id]

    if (
        len(cluster_subset["Optimal_Lag"].unique()) > 1
        and len(cluster_subset) > 2
    ):
        group_model = ols(
            "Fjord_Length ~ C(Optimal_Lag)", data=cluster_subset
        ).fit()
        group_r2_map[cluster_id] = group_model.rsquared
        print(
            f"Group {cluster_id}     (n={len(cluster_subset)}): Categorical R² ="
            f" {group_model.rsquared:.4f}"
        )
    else:
        group_r2_map[cluster_id] = None
        reason = (
            "No Lag Variance (All points in same lag)"
            if len(cluster_subset["Optimal_Lag"].unique()) == 1
            else "Too few points"
        )
        print(
            f"Group {cluster_id}     (n={len(cluster_subset)}): Categorical R² ="
            f" N/A ({reason})"
        )

print("-" * 65)

# ============================================================
# 5. SINGLE PLOT GENERATION (ALL GROUPS COMBINED)
# ============================================================
all_lags = sorted(list(plot_df["Optimal_Lag"].unique()))
if not all_lags:
    all_lags = [0, 1, 2, 3]

fig, ax = plt.subplots(figsize=(12, 8))

# Offset parameters to prevent overlapping markers on identical lags
np.random.seed(42)
cluster_ids = sorted(group_color_map.keys())
num_clusters = len(cluster_ids)
offset_step = 0.08

for cluster_id in cluster_ids:
    cluster_subset = plot_df[plot_df["Cluster"] == cluster_id]

    if len(cluster_subset) > 0:
        # Systematic horizontal offset based on Group ID + subtle jitter
        base_offset = (cluster_id - (num_clusters + 1) / 2) * offset_step
        jitter = np.random.uniform(-0.02, 0.02, size=len(cluster_subset))
        x_pos = cluster_subset["Optimal_Lag"] + base_offset + jitter

        r2_val = group_r2_map.get(cluster_id)
        r2_str = f", $R^2$={r2_val:.2f}" if r2_val is not None else ""
        label_text = f"Group {cluster_id} (n={len(cluster_subset)}{r2_str})"

        ax.scatter(
            x_pos,
            cluster_subset["Fjord_Length"],
            color=group_color_map[cluster_id],
            marker="o",
            s=110,
            edgecolors="black",
            linewidths=0.8,
            alpha=0.9,
            zorder=3,
            label=label_text,
        )

# Title & Formatting
global_r2_str = (
    f" (Global $R^2$ = {global_r2:.3f})" if global_r2 is not None else ""
)
ax.set_title(
    f"Fjord Length vs. Dominant Positive Ocean Lag Across All Groups{global_r2_str}",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

ax.set_xticks(all_lags)
ax.set_xticklabels(
    [f"Lag {l}" for l in all_lags], fontsize=11, fontweight="bold"
)
ax.set_xlim(-0.6, max(all_lags) + 0.6)

ax.set_xlabel(
    "Dominant Positive Ocean Lag", fontsize=12, fontweight="bold", labelpad=10
)
ax.set_ylabel("Fjord Length (m)", fontsize=12, fontweight="bold", labelpad=10)

ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.grid(axis="x", linestyle=":", alpha=0.3)

ax.legend(
    title="Glacier Groups",
    title_fontsize="11",
    loc="upper left",
    frameon=True,
    facecolor="white",
    framealpha=0.9,
    fontsize=10,
)

plt.tight_layout()
plt.show()

In [ ]:
# Subglacial topography across glacier centrelines

import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from google.colab import drive

# 1. MOUNT & LOAD
drive.mount('/content/drive', force_remount=True)
file_path = "/content/drive/MyDrive/Spreadsheets/Subglacial Topography.xlsx"
df = pd.read_excel(file_path)

df = df.rename(columns={
    "Distance along Flowline (m)": "dist",
    "Elevation (m)": "elev",
    "ID": "id"
})

# 2. MAPPING & CLUSTERS
real_glacier_ids = [
    224, 226, 227, 232, 233, 238, 239, 245, 258, 266,
    270, 281, 284, 286, 293, 296, 297, 298, 300, 302,
    303, 304, 310, 320, 323, 330, 335
]
spreadsheet_ids = sorted(df["id"].unique())
id_mapping = dict(zip(spreadsheet_ids, real_glacier_ids))
df["real_id"] = df["id"].map(id_mapping)

cluster_dict = {
    239:1, 245:1, 238:1, 286:1, 323:1, 320:1, 303:1, 300:1, 330:1,
    270:2, 258:2, 281:2, 233:2, 224:2,
    232:3, 266:3, 335:3, 310:3, 296:3, 293:3, 284:3, 302:3, 304:3,
    226:4, 227:4,
    297:5, 298:5
}

df["cluster"] = df["real_id"].map(cluster_dict)
df = df.dropna(subset=["cluster"])
max_limit = 5000
df_filtered = df[df["dist"] <= max_limit].copy()

# 3. COLOUR PALETTE
group_colors = {
    1: "#000000", 2: "#CC79A7", 3: "#D55E00", 4: "#0072B2", 5: "#D4AC0D"
}

# 4. PLOTTING STACKED SUBPLOTS
clusters_to_plot = sorted(df_filtered["cluster"].unique())
n_clusters = len(clusters_to_plot)

fig, axes = plt.subplots(n_clusters, 1, figsize=(10, 2.5 * n_clusters), sharex=True)

if n_clusters == 1:
    axes = [axes]

for i, cluster in enumerate(clusters_to_plot):
    ax = axes[i]
    c_data = df_filtered[df_filtered["cluster"] == cluster]
    glaciers_in_cluster = c_data["real_id"].unique()
    base_color = group_colors.get(cluster, "tab:grey")

    cluster_elevations_for_mean = []
    max_distances_in_group = []

    # Find min/max for dynamic Y-axis stretching
    group_min_y = c_data["elev"].min()
    group_max_y = c_data["elev"].max()
    y_buffer = (group_max_y - group_min_y) * 0.1

    # Add Sea Level Reference
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.2, zorder=1)

    # Plot individual glacier lines (Uniform mid-shade)
    for gid in glaciers_in_cluster:
        g_data = c_data[c_data["real_id"] == gid].sort_values("dist")

        ax.plot(g_data["dist"]/1000, g_data["elev"],
                color=base_color,
                alpha=0.5,          # Mid-shade transparency
                linewidth=1.2,
                zorder=2)

        max_distances_in_group.append(g_data["dist"].max())

    # CALCULATE SHARED MEAN
    shortest_dist = min(max_distances_in_group)
    grid_spacing = 10
    shared_grid = np.arange(0, shortest_dist + grid_spacing, grid_spacing)

    for gid in glaciers_in_cluster:
        g_data = c_data[c_data["real_id"] == gid].sort_values("dist")
        interp_func = interp1d(g_data["dist"], g_data["elev"], bounds_error=False)
        cluster_elevations_for_mean.append(interp_func(shared_grid))

    # Plot the Mean Line (Thick, dark version of same color)
    mean_topo = np.mean(cluster_elevations_for_mean, axis=0)
    ax.plot(shared_grid/1000, mean_topo,
            color=base_color,
            linewidth=2.5,          # Thicker line
            alpha=1.0,             # Full opacity makes it appear darker
            linestyle='--',
            zorder=4)

    # DYNAMIC AXIS FORMATTING
    ax.set_ylim(group_min_y - y_buffer, group_max_y + y_buffer)
    ax.set_ylabel("Elev (m)", fontsize=10, fontweight='bold')
    ax.set_title(f"Group {int(cluster)}", loc='left', fontsize=11, fontweight='bold', color='black', pad=-15)
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.set_xlim(0, max_limit/1000)

# FINAL FORMATTING
axes[-1].set_xlabel("Distance along flowline (km)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Glacier width profiles

import os
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive

# Set default tick label font size (10 * 1.5 = 15)
plt.rcParams.update({"font.size": 15})

# ================================
# 1. MOUNT & SETUP
# ================================
drive.mount("/content/drive", force_remount=True)

save_path = "/content/drive/MyDrive/Figures"
if not os.path.exists(save_path):
    os.makedirs(save_path)

# ================================
# 2. DEFINITIONS & MAPPINGS (UPDATED 6-GROUP SCHEME)
# ================================
real_glacier_ids = [
    224,
    226,
    227,
    232,
    233,
    238,
    239,
    245,
    258,
    266,
    270,
    281,
    284,
    286,
    293,
    296,
    297,
    298,
    300,
    302,
    303,
    304,
    310,
    320,
    323,
    330,
    335,
]

cluster_dict = {
    # Group 1 (11 glacier entries)
    232: 1,
    245: 1,
    296: 1,
    266: 1,
    284: 1,
    286: 1,
    293: 1,
    304: 1,
    300: 1,
    335: 1,
    310: 1,
    # Group 2 (7 glacier entries)
    303: 2,
    320: 2,
    323: 2,
    302: 2,
    330: 2,
    281: 2,
    258: 2,
    # Group 3 (2 glaciers)
    298: 3,
    297: 3,
    # Group 4 (2 glaciers)
    224: 4,
    233: 4,
    # Group 5 (2 glaciers)
    226: 5,
    227: 5,
    # Group 6 (3 glaciers)
    238: 6,
    239: 6,
    270: 6,
}

# ================================
# 3. LOAD DATA
# ================================
file_path = "/content/drive/MyDrive/Spreadsheets/Glacier_Width_Profiles.xlsx"

df_width = pd.read_excel(file_path, sheet_name="Sheet1")
df_width.columns = df_width.columns.str.strip()
df_width = df_width.rename(
    columns={
        "Glacier_ID": "id",
        "Width (m)": "width",
        "Dist_along_flowline (m)": "dist_relative",
        "Original_route_locator": "dist_original",
    }
)

df_term = pd.read_excel(file_path, sheet_name="Terminus_Locations")
df_term.columns = df_term.columns.str.strip()
df_term = df_term.rename(
    columns={"Glacier_ID": "id", "Original_route_locator": "dist_original"}
)

date_col = df_term.columns[2]
df_term["year"] = pd.to_datetime(df_term[date_col], dayfirst=True).dt.year

spreadsheet_ids = sorted(df_width["id"].unique())
id_mapping = dict(zip(spreadsheet_ids, real_glacier_ids))
df_width["real_id"] = df_width["id"].map(id_mapping)
df_term["real_id"] = df_term["id"].map(id_mapping)

df_width["cluster"] = df_width["real_id"].map(cluster_dict)
df_width = df_width.dropna(subset=["cluster"])

# ================================
# 4. GLOBAL COLOR SETUP (VIRIDIS)
# ================================
sorted_years = sorted(df_term["year"].unique())
min_year, max_year = min(sorted_years), max(sorted_years)
norm = mcolors.Normalize(vmin=min_year, vmax=max_year)
universal_cmap = plt.get_cmap("viridis")

# ================================
# 5. GRID LAYOUT & PLOTTING
# ================================
max_glaciers = max(
    len([g for g, c in cluster_dict.items() if c == i]) for i in range(1, 7)
)
n_rows = 4

fig = plt.figure(figsize=(4.2 * max_glaciers + 2, 32))


def plot_glacier_subplot(ax, gid, is_first_col=False):
    g_data = df_width[df_width["real_id"] == gid].sort_values("dist_relative")
    g_term = df_term[df_term["real_id"] == gid].copy()

    if g_data.empty:
        ax.axis("off")
        return

    anchor_orig = g_data["dist_original"].iloc[0]
    anchor_rel = g_data["dist_relative"].iloc[0]
    g_term["rel_pos"] = anchor_rel + (anchor_orig - g_term["dist_original"])

    prof_min_km = g_data["dist_relative"].min() / 1000
    prof_max_km = g_data["dist_relative"].max() / 1000

    # Background shading
    ax.axhspan(-100, prof_min_km, color="lightblue", alpha=0.4, zorder=0)
    ax.axhspan(prof_max_km, 200, color="gainsboro", alpha=0.5, zorder=0)
    ax.axvline(0, color="black", linestyle="-", linewidth=0.8, alpha=0.3)

    # Width profile
    if gid == 233:
        meas_w = g_data["width"].iloc[0]
        meas_dist = g_data["dist_relative"].iloc[0] / 1000
        ax.plot(
            [-meas_w / 2, meas_w / 2],
            [meas_dist, meas_dist],
            color="tab:grey",
            linestyle="--",
            linewidth=2,
            zorder=3,
        )
        ax.plot(
            [-meas_w / 2, meas_w / 2],
            [meas_dist, meas_dist],
            marker="*",
            color="tab:grey",
            linestyle="None",
            markersize=10,
            zorder=4,
        )
    else:
        ax.plot(
            g_data["width"] / 2,
            g_data["dist_relative"] / 1000,
            color="tab:grey",
            linewidth=2,
            zorder=3,
        )
        ax.plot(
            -g_data["width"] / 2,
            g_data["dist_relative"] / 1000,
            color="tab:grey",
            linewidth=2,
            zorder=3,
        )

    # Terminus positions (Viridis scale)
    years = sorted(g_term["year"].unique())
    for year in years:
        t_pos_m = g_term[g_term["year"] == year]["rel_pos"].iloc[0]
        t_pos_km = t_pos_m / 1000
        year_color = universal_cmap(norm(year))

        if gid == 233:
            ax.scatter(
                0,
                t_pos_km,
                color=year_color,
                s=100,
                edgecolors="black",
                linewidths=0.5,
                zorder=10,
            )
        else:
            if prof_min_km <= t_pos_km <= prof_max_km:
                idx_nearest = (g_data["dist_relative"] - t_pos_m).abs().idxmin()
                w = g_data.loc[idx_nearest, "width"]
                ax.hlines(
                    y=t_pos_km,
                    xmin=-w / 2,
                    xmax=w / 2,
                    color=year_color,
                    linewidth=2,
                    zorder=5,
                )
            else:
                ax.scatter(
                    0,
                    t_pos_km,
                    color=year_color,
                    s=80,
                    edgecolors="black",
                    linewidths=0.5,
                    zorder=6,
                )

    # Formatting (scaled 1.5x)
    ax.set_title(f"{gid}", fontsize=28, fontweight="bold")
    ax.set_xlabel("Width (m)", fontsize=20)
    if is_first_col:
        ax.set_ylabel("Distance (km)", fontsize=22, fontweight="bold")

    ax.tick_params(axis="both", which="major", labelsize=20)

    y_min = min(prof_min_km, g_term["rel_pos"].min() / 1000) - 0.5
    y_max = max(prof_max_km, g_term["rel_pos"].max() / 1000) + 0.5
    ax.set_ylim(y_min, y_max)
    ax.grid(axis="y", linestyle="--", alpha=0.2)


# --- ROW 1: GROUP 1 ---
g1_glaciers = sorted([gid for gid, c in cluster_dict.items() if c == 1])
fig.text(0.01, 0.86, "Group 1", fontsize=36, fontweight="bold", rotation=90, va="center")
for idx, gid in enumerate(g1_glaciers):
    ax = plt.subplot(n_rows, max_glaciers, idx + 1)
    plot_glacier_subplot(ax, gid, is_first_col=(idx == 0))

# --- ROW 2: GROUP 2 ---
g2_glaciers = sorted([gid for gid, c in cluster_dict.items() if c == 2])
fig.text(0.01, 0.61, "Group 2", fontsize=36, fontweight="bold", rotation=90, va="center")
for idx, gid in enumerate(g2_glaciers):
    ax = plt.subplot(n_rows, max_glaciers, max_glaciers + idx + 1)
    plot_glacier_subplot(ax, gid, is_first_col=(idx == 0))

# --- ROW 3: SHARED BY GROUP 3 AND GROUP 4 ---
g3_glaciers = sorted([gid for gid, c in cluster_dict.items() if c == 3])
g4_glaciers = sorted([gid for gid, c in cluster_dict.items() if c == 4])

row3_base_idx = 2 * max_glaciers
fig.text(0.01, 0.36, "Group 3 / Group 4", fontsize=36, fontweight="bold", rotation=90, va="center")

for col_idx, gid in zip([0, 1], g3_glaciers):
    ax = plt.subplot(n_rows, max_glaciers, row3_base_idx + col_idx + 1)
    plot_glacier_subplot(ax, gid, is_first_col=(col_idx == 0))

for col_idx, gid in zip([3, 4], g4_glaciers):
    ax = plt.subplot(n_rows, max_glaciers, row3_base_idx + col_idx + 1)
    plot_glacier_subplot(ax, gid, is_first_col=False)

# --- ROW 4: SHARED BY GROUP 5 AND GROUP 6 ---
g5_glaciers = sorted([gid for gid, c in cluster_dict.items() if c == 5])
g6_glaciers = sorted([gid for gid, c in cluster_dict.items() if c == 6])

row4_base_idx = 3 * max_glaciers
fig.text(0.01, 0.11, "Group 5 / Group 6", fontsize=36, fontweight="bold", rotation=90, va="center")

for col_idx, gid in zip([0, 1], g5_glaciers):
    ax = plt.subplot(n_rows, max_glaciers, row4_base_idx + col_idx + 1)
    plot_glacier_subplot(ax, gid, is_first_col=(col_idx == 0))

for col_idx, gid in zip([3, 4, 5], g6_glaciers):
    ax = plt.subplot(n_rows, max_glaciers, row4_base_idx + col_idx + 1)
    plot_glacier_subplot(ax, gid, is_first_col=False)

# ================================
# 6. UNIFIED LEGEND & ADJUSTMENTS
# ================================
legend_handles = []
for year in sorted_years:
    line = plt.Line2D([0], [0], color=universal_cmap(norm(year)), linewidth=4)
    legend_handles.append(line)

fig.legend(
    legend_handles,
    [str(y) for y in sorted_years],
    loc="lower right",
    title="Terminus Year",
    bbox_to_anchor=(0.98, 0.01),
    fontsize=30,
    title_fontsize=33,
    labelspacing=0.8,
    ncol=2,
)

plt.tight_layout(rect=[0.05, 0.04, 0.95, 0.96], h_pad=4.0, w_pad=1.5)

plt.savefig(
    os.path.join(save_path, "Glacier_Profiles_Merged_6Groups.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()